In [1]:
!python --version

Python 3.13.2


In [2]:
#importing necessary modules 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from sklearn.decomposition import PCA
#from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import copairs
import pycytominer
from copairs.map import average_precision
from copairs.map import mean_average_precision
from utils import * 

%load_ext autoreload
%autoreload 2

/opt/anaconda3/envs/copairs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pip show copairs

Name: copairs
Version: 0.5.1
Summary: Find pairs and compute metrics between them
Home-page: https://github.com/cytomining/copairs
Author: 
Author-email: John Arevalo <johnarevalo@gmail.com>, Alexandr Kalinin <akalinin@broadinstitute.org>, "Alan F. Munoz" <amunozgo@broadinstitute.org>
License: 
Location: /opt/anaconda3/envs/copairs/lib/python3.13/site-packages
Requires: duckdb, pandas, statsmodels, tqdm
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
#Reading the normalized and feature selected files for copairs - fixed cells and 48h time point
profiles = {'/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122250_normalized_feature_select_negcon_batch.csv':'Standard CP',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122246_normalized_feature_select_negcon_batch.csv':'CP + MitoBrilliant',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122247_normalized_feature_select_negcon_batch.csv':'CP + Phenovue phalloidin 400LS',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122248_normalized_feature_select_negcon_batch.csv':'Standard CP (exposed to ChromaLive)',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122249_normalized_feature_select_negcon_batch_wo_phasefeatures.csv':'ChromaLive + Hoechst'

}


In [25]:
def cell_counts(input_dict={}):
     
    combined_df = pd.DataFrame()

    for i in input_dict:
            
            with open(i, 'rb') as filetype:
                if filetype.read(2) == b'\x1f\x8b':
                    df = pd.read_csv(i, compression='gzip')
                else:
                    df = pd.read_csv(i)
            subset_df = df[['Metadata_Count_Cells']]
            subset_df = subset_df.rename(columns={'Metadata_Count_Cells': input_dict[i]})
            combined_df = pd.concat([combined_df, subset_df], axis=1)
    combined_df = pd.concat([combined_df, df['Metadata_Well']], axis=1)

    return combined_df


In [41]:
cell_count_df = cell_counts(profiles)

In [42]:
cell_count_df

,Standard CP,CP + MitoBrilliant,CP + Phenovue phalloidin 400LS,Standard CP (exposed to ChromaLive),ChromaLive + Hoechst,Metadata_Well
0,3121,3456.0,3361,3093,3219,A01
1,3804,4102.0,4012,3539,3824,A02
2,3766,3903.0,4048,3567,3838,A03
3,2922,3304.0,3239,2711,2768,A04
4,3971,4045.0,3955,3502,3722,A05
...,...,...,...,...,...,...
379,4017,3614.0,3836,3428,3445,P20
380,3634,3995.0,3505,3094,3593,P21
381,3835,4099.0,3974,3041,3752,P22
382,4043,3411.0,4048,3336,3628,P23


In [44]:
cell_count_df.to_csv('/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/cell_counts.csv', index=False)

### Standard CP

In [28]:
cell_count_df['Standard CP'].min()

np.int64(58)

In [30]:
cell_count_df['Standard CP'].max()

np.int64(4746)

### CP + MitoBrilliant

In [32]:
cell_count_df['CP + MitoBrilliant'].min()

np.float64(42.0)

In [33]:
cell_count_df['CP + MitoBrilliant'].max()

np.float64(4882.0)

### CP + Phenovue phalloidin 400LS

In [34]:
cell_count_df['CP + Phenovue phalloidin 400LS'].min()

np.int64(40)

In [35]:
cell_count_df['CP + Phenovue phalloidin 400LS'].max()

np.int64(4624)

### Standard CP (exposed to ChromaLive)	

In [36]:
cell_count_df['Standard CP (exposed to ChromaLive)'].min()

np.int64(84)

In [37]:
cell_count_df['Standard CP (exposed to ChromaLive)'].max()

np.int64(4529)

### ChromaLive + Hoechst

In [38]:
cell_count_df['ChromaLive + Hoechst'].min()

np.int64(236)

In [39]:
cell_count_df['ChromaLive + Hoechst'].max()

np.int64(4791)